# 03 - Delivery Lateness vs. Review Score (Statistical Test)

**Question:** do late deliveries receive lower review scores than on-time deliveries?

**H0:** the review-score distributions of late and on-time orders do not differ.
**H1 (one-sided):** late orders' review scores are stochastically lower than on-time orders'.

**Why Mann-Whitney U, not a t-test:** review scores are ordinal (1-5) and heavily skewed
toward 5, so comparing means via a t-test assumes an interval scale and near-normality -
the textbook-wrong choice here. Mann-Whitney U compares the two distributions directly
without that assumption (MASTER_DOC section 13).

In [1]:
import sys

sys.path.insert(0, "..")

import pandas as pd
from scipy.stats import mannwhitneyu
from sqlalchemy import text

from src.utils import get_engine

engine = get_engine()

In [2]:
df = pd.read_sql(
    text("""
        SELECT r.review_score, o.is_late
        FROM analytics.fact_orders o
        JOIN analytics.order_reviews r USING (order_id)
        WHERE o.order_status = 'delivered' AND o.is_late IS NOT NULL
    """),
    engine,
)
print(f"n = {len(df)} reviewed, delivered orders with a known lateness flag")
df.groupby("is_late")["review_score"].describe()

n = 95824 reviewed, delivered orders with a known lateness flag


,count,mean,std,min,25%,50%,75%,max
is_late,,,,,,,,
False,88163.0,4.294114,1.147252,1.0,4.0,5.0,5.0,5.0
True,7661.0,2.565070,1.658462,1.0,1.0,2.0,4.0,5.0


## Mann-Whitney U test

In [3]:
late = df.loc[df["is_late"], "review_score"]
on_time = df.loc[~df["is_late"], "review_score"]

stat, p_value = mannwhitneyu(late, on_time, alternative="less")
print(f"n_late={len(late)}, n_on_time={len(on_time)}")
print(f"U statistic = {stat:.1f}")
print(f"p-value (one-sided, late < on_time) = {p_value:.3e}")

n_late=7661, n_on_time=88163
U statistic = 150706850.5
p-value (one-sided, late < on_time) = 0.000e+00


## Reporting standard: p-value + medians + 1-2 star share

In [4]:
summary = df.groupby("is_late")["review_score"].agg(
    n="count", mean="mean", median="median",
).round(3)
summary["pct_1_2_star"] = (
    df.groupby("is_late")["review_score"]
    .apply(lambda s: (s <= 2).mean() * 100)
    .round(2)
)
summary

,n,mean,median,pct_1_2_star
is_late,,,,
False,88163,4.294,5.0,9.22
True,7661,2.565,2.0,54.07


In [5]:
late_row = summary.loc[True]
on_time_row = summary.loc[False]
print(
    f"p-value is effectively 0 (n in the tens of thousands - at this sample size the "
    f"p-value itself is not the insight; the effect size is).\n"
    f"Late deliveries: median review score {late_row['median']:.0f}/5, with "
    f"{late_row['pct_1_2_star']:.1f}% rated 1-2 stars (n={late_row['n']:.0f}).\n"
    f"On-time deliveries: median review score {on_time_row['median']:.0f}/5, with "
    f"{on_time_row['pct_1_2_star']:.1f}% rated 1-2 stars (n={on_time_row['n']:.0f}).\n"
    f"H0 is rejected: late deliveries are associated with materially and significantly "
    f"lower review scores."
)

p-value is effectively 0 (n in the tens of thousands - at this sample size the p-value itself is not the insight; the effect size is).
Late deliveries: median review score 2/5, with 54.1% rated 1-2 stars (n=7661).
On-time deliveries: median review score 5/5, with 9.2% rated 1-2 stars (n=88163).
H0 is rejected: late deliveries are associated with materially and significantly lower review scores.


**Causal humility:** this is an observational association, not proof of causation - late
delivery correlates with other factors (shipping distance, specific sellers, freight cost)
that could themselves affect satisfaction. Stating this signals analytical maturity, not
weakness (MASTER_DOC section 13).